# 03 — Stratified Train / Validation / Test Split

This notebook creates the **canonical data split** that every subsequent experiment must reuse.

Rationale for the split ratios (60 / 20 / 20):
- The dataset is moderately sized (~2 × n_melanoma samples). A 60 % training set gives the GMM and the neural-network head enough data to fit without overfitting.
- Equal validation and test sets (20 % each) ensure that hyperparameter selection (on val) and final evaluation (on test) are estimated with similar statistical precision.
- Stratification preserves the 50 / 50 class balance in every split, so no split accidentally becomes imbalanced.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from config import SEED, PROCESSED_DIR, TABLES_DIR

## 1 — Load features

In [ ]:
data = np.load(PROCESSED_DIR / "features_hsv.npz")
X, y = data["X"], data["y"]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Class counts — melanoma: {(y==1).sum()}, nevus: {(y==0).sum()}")

## 2 — Stratified split

We apply `train_test_split` **twice**:

```
full dataset (100 %)
    └─ test      (20 %)   ← held out now, never touched until final evaluation
    └─ remaining (80 %)
           └─ train (60 % of full = 75 % of remaining)
           └─ val   (20 % of full = 25 % of remaining)
```

Both calls use `stratify=y` and the same `SEED` so the split is fully reproducible.

In [ ]:
# Step 1: carve out test (20 %)
X_rem, X_test, y_rem, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

# Step 2: split remaining 80 % into train (75 %) and val (25 %)
# 0.25 × 0.80 = 0.20 of the full dataset → val is exactly 20 %
X_train, X_val, y_train, y_val = train_test_split(
    X_rem, y_rem, test_size=0.25, stratify=y_rem, random_state=SEED
)

print(f"Train : {X_train.shape}  melanoma={y_train.sum()}  nevus={(y_train==0).sum()}")
print(f"Val   : {X_val.shape}    melanoma={y_val.sum()}  nevus={(y_val==0).sum()}")
print(f"Test  : {X_test.shape}   melanoma={y_test.sum()}  nevus={(y_test==0).sum()}")
print(f"Total : {len(y_train)+len(y_val)+len(y_test)}  (original: {len(y)})")

## 3 — Confirm stratification

The melanoma fraction should be ≈ 0.50 in every split.

In [ ]:
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    frac = yy.mean()
    print(f"  {name:5s}: melanoma fraction = {frac:.4f}")

## 4 — Save splits

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "splits.npz"
np.savez_compressed(
    out_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)
print(f"Splits saved to: {out_path}")

## 5 — Summary table

In [ ]:
rows = []
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    rows.append({
        "split":      name,
        "n_total":    len(yy),
        "n_melanoma": int(yy.sum()),
        "n_benign":   int((yy == 0).sum()),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TABLES_DIR / "split_summary.csv", index=False)
print(f"\nSaved to: {TABLES_DIR / 'split_summary.csv'}")